# Task 2: API REST — RAWG Video Games Database
### Evelyn Valeria Sarmiento Vásquez


In [1]:
!pip install requests pandas


In [ ]:
# Importando las librerías necesarias
import requests
import pandas as pd
from getpass import getpass

# getpass te pide la key como si fuera una contraseña — no queda guardada en el notebook
API_KEY = getpass("Ingresa tu RAWG API Key: ")

# URL base de la API — todas las consultas empiezan con esto
BASE_URL = "https://api.rawg.io/api"

#Al correr esto, va a aparecer un espacio para imputar mi api key como contraseña, es decir **********

print("API Key ingresada correctamente :)")


In [3]:
# Creamos un cliente que maneja todas las llamadas a la API
# Su función principal es dos cosas:
#   1. Agregar automáticamente tu API key a cada consulta (sin que tengas que escribirla cada vez)
#   2. Contar cuántas llamadas haces en total (para responder la pregunta D1)

class RAWGClient:
    def __init__(self, api_key):
        self.api_key = api_key
        self.total_requests = 0  # Contador de llamadas

    def get(self, endpoint, params={}):
        # Agregamos la API key a los parámetros automáticamente
        params["key"] = self.api_key
        url = f"{BASE_URL}/{endpoint}"
        response = requests.get(url, params=params)
        self.total_requests += 1  # Sumamos 1 al contador
        return response.json()

    def resumen_requests(self):
        print(f"Total de llamadas a la API realizadas: {self.total_requests}")

# Creamos el cliente listo para usar
client = RAWGClient(API_KEY)
print("Cliente RAWG listo ✓")


Cliente RAWG listo ✓


## Parte A — Exploración General

En esta sección consultamos la API de RAWG para obtener información general
sobre su base de datos de videojuegos.


In [4]:
# A1: Total de juegos registrados en RAWG
# El endpoint /games devuelve en el campo "count" el total de juegos en la base de datos

data = client.get("games")
total_games = data["count"]

print(f"RAWG tiene registrados {total_games:,} juegos en total.")


RAWG tiene registrados 898,445 juegos en total.


## Parte B — Análisis por Categorías

En esta sección analizamos los juegos mejor valorados según Metacritic,
y los mejores juegos disponibles en Steam.


In [5]:
# B1: Top 5 juegos con mayor puntaje Metacritic de todos los tiempos
# Ordenamos por metacritic de mayor a menor con el parámetro ordering=-metacritic

data = client.get("games", params={
    "ordering": "-metacritic",  # - significa descendente (mayor a menor)
    "page_size": 5              # Solo queremos los 5 primeros
})

print("🏆 Top 5 juegos mejor valorados por Metacritic:\n")
for i, game in enumerate(data["results"], start=1):
    print(f"{i}. {game['name']}")
    print(f"   Rating: {game['rating']} | Metacritic: {game['metacritic']}")
    print()


🏆 Top 5 juegos mejor valorados por Metacritic:

1. The Legend of Zelda: Ocarina of Time
   Rating: 4.38 | Metacritic: 99

2. Soulcalibur (1998)
   Rating: 0.0 | Metacritic: 98

3. Soulcalibur
   Rating: 4.38 | Metacritic: 98

4. Baldur's Gate III
   Rating: 4.44 | Metacritic: 97

5. Metroid Prime
   Rating: 4.35 | Metacritic: 97



In [6]:
# B2: Top 10 mejores juegos disponibles en Steam (store_id=1)
# Filtramos por la tienda Steam y ordenamos por metacritic

data = client.get("games", params={
    "stores": 1,                # store_id=1 es Steam
    "ordering": "-metacritic",
    "page_size": 10
})

print("🎮 Top 10 mejores juegos en Steam:\n")
for i, game in enumerate(data["results"], start=1):
    print(f"{i}. {game['name']}")
    print(f"   Rating: {game['rating']} | Metacritic: {game['metacritic']}")
    print()


🎮 Top 10 mejores juegos en Steam:

1. Baldur's Gate III
   Rating: 4.44 | Metacritic: 97

2. Half-Life 2: Update
   Rating: 4.13 | Metacritic: 96

3. Half-Life
   Rating: 4.38 | Metacritic: 96

4. Red Dead Redemption 2
   Rating: 4.59 | Metacritic: 96

5. Half-Life 2
   Rating: 4.48 | Metacritic: 96

6. BioShock
   Rating: 4.36 | Metacritic: 96

7. Grand Theft Auto IV: Complete Edition
   Rating: 4.57 | Metacritic: 95

8. Divinity: Original Sin 2
   Rating: 4.38 | Metacritic: 95

9. Portal 2
   Rating: 4.58 | Metacritic: 95

10. Red Dead Redemption
   Rating: 4.42 | Metacritic: 95



## Parte C — Comparaciones

En esta sección comparamos juegos por plataforma, géneros y años de lanzamiento.


In [7]:
# C1: Top 5 juegos en PC (platform_id=4) vs top 5 en PS5 (platform_id=187)
# Comparamos qué plataforma tiene los juegos mejor valorados

pc_data = client.get("games", params={
    "platforms": 4,
    "ordering": "-metacritic",
    "page_size": 5
})

ps5_data = client.get("games", params={
    "platforms": 187,
    "ordering": "-metacritic",
    "page_size": 5
})

print("🖥️  Top 5 PC:\n")
pc_ratings = []
for i, game in enumerate(pc_data["results"], start=1):
    print(f"  {i}. {game['name']} | Metacritic: {game['metacritic']}")
    pc_ratings.append(game["metacritic"] or 0)

print(f"\n  Promedio Metacritic PC: {sum(pc_ratings)/len(pc_ratings):.1f}")

print("\n🎮 Top 5 PS5:\n")
ps5_ratings = []
for i, game in enumerate(ps5_data["results"], start=1):
    print(f"  {i}. {game['name']} | Metacritic: {game['metacritic']}")
    ps5_ratings.append(game["metacritic"] or 0)

print(f"\n  Promedio Metacritic PS5: {sum(ps5_ratings)/len(ps5_ratings):.1f}")

winner = "PC" if sum(pc_ratings) > sum(ps5_ratings) else "PS5"
print(f"\n🏆 La plataforma con juegos mejor valorados es: {winner}")


🖥️  Top 5 PC:

  1. Baldur's Gate III | Metacritic: 97
  2. Half-Life 2: Update | Metacritic: 96
  3. Half-Life | Metacritic: 96
  4. Red Dead Redemption 2 | Metacritic: 96
  5. Half-Life 2 | Metacritic: 96

  Promedio Metacritic PC: 96.2

🎮 Top 5 PS5:

  1. Baldur's Gate III | Metacritic: 97
  2. Red Dead Redemption | Metacritic: 95
  3. Elden Ring | Metacritic: 95
  4. The Elder Scrolls V: Skyrim | Metacritic: 94
  5. Quake | Metacritic: 94

  Promedio Metacritic PS5: 95.0

🏆 La plataforma con juegos mejor valorados es: PC


In [8]:
# C2: Comparación de 3 juegos famosos

# Exploramos los juegos más populares para elegir cuáles comparar
data = client.get("games", params={
    "ordering": "-metacritic",
    "page_size": 20
})

print("Juegos famosos disponibles en RAWG:\n")
for i, game in enumerate(data["results"], start=1):
    genres = ", ".join([g["name"] for g in game["genres"]])
    print(f"{i:>2}. {game['name']}")
    print(f"     Metacritic: {game['metacritic']} | Rating: {game['rating']} | Géneros: {genres}")
    print()



Juegos famosos disponibles en RAWG:

 1. The Legend of Zelda: Ocarina of Time
     Metacritic: 99 | Rating: 4.38 | Géneros: Action, Adventure, RPG

 2. Soulcalibur (1998)
     Metacritic: 98 | Rating: 0.0 | Géneros: Fighting

 3. Soulcalibur
     Metacritic: 98 | Rating: 4.38 | Géneros: Action, Fighting

 4. Baldur's Gate III
     Metacritic: 97 | Rating: 4.44 | Géneros: Adventure, RPG, Strategy

 5. Metroid Prime
     Metacritic: 97 | Rating: 4.35 | Géneros: Action, Shooter, Adventure

 6. Perfect Dark
     Metacritic: 97 | Rating: 3.98 | Géneros: Action, Shooter

 7. Super Mario Odyssey
     Metacritic: 97 | Rating: 4.42 | Géneros: Arcade, Platformer

 8. Super Mario Galaxy 2
     Metacritic: 97 | Rating: 4.34 | Géneros: Platformer

 9. Super Mario Galaxy
     Metacritic: 97 | Rating: 4.35 | Géneros: Platformer

10. The Legend of Zelda: Breath of the Wild
     Metacritic: 97 | Rating: 4.47 | Géneros: Action, Adventure, RPG

11. Half-Life 2: Update
     Metacritic: 96 | Rating: 4.13 |

In [9]:
# C2: Comparación de 3 entregas de la saga The Legend of Zelda
# Buscamos cada juego por nombre y mostramos sus datos en una tabla

games_to_compare = [
    "The Legend of Zelda: Ocarina of Time",
    "The Legend of Zelda: Breath of the Wild",
    "The Legend of Zelda: Tears of the Kingdom"
]

rows = []
for game_name in games_to_compare:
    data = client.get("games", params={"search": game_name, "page_size": 1})
    game = data["results"][0]

    genres = ", ".join([g["name"] for g in game["genres"]])
    platforms = ", ".join([p["platform"]["name"] for p in game["platforms"]])

    rows.append({
        "Nombre": game["name"],
        "Rating": game["rating"],
        "Metacritic": game["metacritic"],
        "Géneros": genres,
        "Plataformas": platforms
    })

df_compare = pd.DataFrame(rows)
df_compare


,Nombre,Rating,Metacritic,Géneros,Plataformas
0,The Legend of Zelda: Ocarina of Time,4.38,99,"Adventure, Action, RPG","Nintendo Switch, Nintendo 64"
1,The Legend of Zelda: Breath of the Wild,4.47,97,"Adventure, Action, RPG","Nintendo Switch, Wii U"
2,The Legend of Zelda: Tears of the Kingdom,4.38,96,"Adventure, Action",Nintendo Switch


In [13]:
# C3: Top 5 juegos de al menos 4 géneros distintos
# Primero exploramos qué géneros tienen más juegos registrados en RAWG
# para asegurarnos de elegir géneros con suficiente data

genres_data = client.get("genres", params={"page_size": 20})

print("Géneros disponibles en RAWG ordenados por cantidad de juegos:\n")
genres_list = sorted(genres_data["results"], key=lambda x: x["games_count"], reverse=True)

for i, genre in enumerate(genres_list[:10], start=1):
    print(f"  {i:>2}. {genre['name']:<20} → {genre['games_count']:,} juegos | slug: {genre['slug']}")



Géneros disponibles en RAWG ordenados por cantidad de juegos:

   1. Action               → 191,565 juegos | slug: action
   2. Adventure            → 151,802 juegos | slug: adventure
   3. Platformer           → 100,929 juegos | slug: platformer
   4. Puzzle               → 97,421 juegos | slug: puzzle
   5. Indie                → 86,376 juegos | slug: indie
   6. Simulation           → 76,880 juegos | slug: simulation
   7. Casual               → 67,663 juegos | slug: casual
   8. Strategy             → 62,408 juegos | slug: strategy
   9. RPG                  → 61,957 juegos | slug: role-playing-games-rpg
  10. Shooter              → 59,634 juegos | slug: shooter


In [14]:
# C3: Top 5 juegos de los 5 géneros con más juegos registrados en RAWG
# Calculamos el promedio de rating por género para ver cuál produce los mejores juegos

genres_to_check = ["action", "adventure", "platformer", "puzzle", "indie"]
genre_averages = {}

for genre in genres_to_check:
    data = client.get("games", params={
        "genres": genre,
        "ordering": "-metacritic",
        "page_size": 5
    })
    ratings = [g["rating"] for g in data["results"] if g["rating"]]
    avg = sum(ratings) / len(ratings) if ratings else 0
    genre_averages[genre] = round(avg, 2)
    print(f"  {genre.capitalize():<15} promedio rating = {avg:.2f}")

best_genre = max(genre_averages, key=genre_averages.get)
print(f"\n🏆 El género con mejor rating promedio es: {best_genre.capitalize()} ({genre_averages[best_genre]})")


  Action          promedio rating = 4.31
  Adventure       promedio rating = 4.40
  Platformer      promedio rating = 4.29
  Puzzle          promedio rating = 4.25
  Indie           promedio rating = 4.22

🏆 El género con mejor rating promedio es: Adventure (4.4)


In [15]:
# C4: Mejores juegos de 3 años distintos
# Comparamos un año pre pandemia (2018), durante la pandemia (2020) y post pandemia (2023)

years = [2018, 2020, 2023]
year_averages = {}

for year in years:
    data = client.get("games", params={
        "dates": f"{year}-01-01,{year}-12-31",
        "ordering": "-metacritic",
        "page_size": 5
    })
    scores = [g["metacritic"] for g in data["results"] if g["metacritic"]]
    avg = sum(scores) / len(scores) if scores else 0
    year_averages[year] = round(avg, 2)
    print(f"  {year}: promedio Metacritic = {avg:.2f}")

best_year = max(year_averages, key=year_averages.get)
print(f"\n🏆 El año con juegos mejor valorados es: {best_year} ({year_averages[best_year]})")


  2018: promedio Metacritic = 93.60
  2020: promedio Metacritic = 93.00
  2023: promedio Metacritic = 92.40

🏆 El año con juegos mejor valorados es: 2018 (93.6)


In [ ]:
# C5: Exportamos los 20 mejores juegos de todos los tiempos a un CSV
#Definimos "mejor" según Metacritic - métrica usada por críticos profesionales
# El CSV se guarda en api/output/top20_rawg.csv

import os 

data = client.get("games", params={
    "ordering": "-metacritic",
    "page_size": 20
})

rows = []
for game in data["results"]:
    main_genre = game["genres"][0]["name"] if game["genres"] else "N/A"
    rows.append({
        "name": game["name"],
        "rating": game["rating"],
        "metacritic": game["metacritic"],
        "release_date": game["released"],
        "main_genre": main_genre
    })

df_top20 = pd.DataFrame(rows)

os.makedirs("output", exist_ok=True)
df_top20.to_csv("output/top20_rawg.csv", index=False)

print("Archivo guardado: output/top20_rawg.csv")
print(f"Total filas: {len(df_top20)}\n")
df_top20.head()


Archivo guardado: output/top20_rawg.csv
Total filas: 20



,name,rating,metacritic,release_date,main_genre
0,The Legend of Zelda: Ocarina of Time,4.38,99,1998-11-21,Action
1,Soulcalibur (1998),0.00,98,1998-07-30,Fighting
2,Soulcalibur,4.38,98,1998-07-30,Action
3,Baldur's Gate III,4.44,97,2023-08-03,Adventure
4,Metroid Prime,4.35,97,2002-11-17,Action
